In [31]:
from pathlib import Path
import numpy as np
from plant_shape_analysis import LeafSequencesDataset
from plant_shape_analysis.vis.plot_functions import visualize_leaf_sequence

In [32]:
selected_sequences = [
"maize_control_plant1", # OK
"maize_control_plant2", # OK
"maize_control_plant3", # OK
"sorghum_control_plant1", # OK
"sorghum_control_plant2", # OK
"sorghum_highlight_plant2", # OK
"tobacco_control_plant1", # OK, FIXED alignment issue (last timestep)
"tobacco_control_plant2", # OK
"tobacco_shade_plant3", # OK, FIXED alignment issue (first 4 timesteps) and wrong labels
"tomato1_heat_plant3", # OK
"tomato1_shade_plant1", # OK
"tomato2_control_plant3" # OK
]

### Get data

In [33]:
# Load dataset
dataset_path = Path("../data/TrackPlant3D/versions")
leaf_dataset = LeafSequencesDataset(
    dataset_path,
    use_ply=True,
    version="v2",
    apply_alignment=True,
    plant_alignment_method=None,
    estimate_plant_normals=False,
    auto_download=False,
)

Apply multi-stage PCA-based alignment to 218 leaf sequences...
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first princ

In [34]:
# Visualize all leaves of selected sequences
selected_leaves = []
for plant_name in sorted(selected_sequences):
    all_leaves = leaf_dataset.get_timeseries_by_plant_sequence(plant_name)
    selected_leaves.extend([leaf["sequence_name"] for leaf in all_leaves])
selected_leaves[:5], len(selected_leaves) # 55 leaf sequences to inspect

(['maize_control_plant1_leaf1',
  'maize_control_plant1_leaf2',
  'maize_control_plant1_leaf3',
  'maize_control_plant2_leaf1',
  'maize_control_plant2_leaf2'],
 55)

#### Workflow

In [36]:
# Visualize leaf to identify problematic timepoints
#leaf_name = "maize_control_plant2_leaf3"
leaf_name = selected_leaves[5]  # Change index to visualize different leaves
leaf_seq = leaf_dataset.get_timeseries_by_sequence_name(leaf_name)
visualize_leaf_sequence(leaf_seq, window_name=leaf_name, show_coordinate_frame=True)

In [ ]:
# Visualize leaf and note problematic timepoints
leaf_seq = leaf_dataset.get_timeseries_by_sequence_name(leaf_name)
visualize_leaf_sequence(leaf_seq, window_name=leaf_name, show_coordinate_frame=True)

In [ ]:
# Apply manual corrections
# List the timepoint INDICES (0-based) that need normal flipping
# Example: if timepoint 1 and 3 are wrong, use [1, 3]
timepoints_to_flip = [0]  # MODIFY THIS based on visual inspection

# Apply the flip
leaf_dataset.flip_normals_for_timepoints(leaf_name, timepoints_to_flip)

# Visualize leaf again
visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_name}_corrected", show_coordinate_frame=True)

✓ Flipped normals for maize_control_plant2_leaf3 timepoint 0 (day 0)

Updated normal Z-component for each timepoint:
  ✗ t=0 day=0: mean_z=0.0038
  ✓ t=1 day=1: mean_z=-0.0381
  ✓ t=2 day=2: mean_z=-0.0190
  ✓ t=3 day=3: mean_z=-0.0860
  ✓ t=4 day=4: mean_z=-0.0091
  ✗ t=5 day=5: mean_z=0.0126
  ✓ t=6 day=6: mean_z=-0.0007
  ✗ t=7 day=7: mean_z=0.0009
  ✗ t=8 day=8: mean_z=0.0120


In [30]:
# Save corrected leaf back to plant PLY
# This maps the corrected normals back to the plant and saves PLY files
output_dir = dataset_path / "v2" / "gt_corrected_v2"
leaf_dataset.save_leaf_to_plant_ply(leaf_name, output_dir=output_dir)

✓ Saved updated plant PLY files to: ../data/TrackPlant3D/versions/v2/gt_corrected_v2


PosixPath('../data/TrackPlant3D/versions/v2/gt_corrected_v2')

In [ ]:
# (Optional): Batch process multiple leaves
# If you have multiple leaves from the same plant that need correction

corrections = {
    "maize_control_plant2_leaf3": [1, 2],     # flip timepoints 1 and 2
    "maize_control_plant2_leaf4": [0, 3, 4],  # flip timepoints 0, 3, and 4
    # Add more as needed...
}

for leaf_name, flip_indices in corrections.items():
    print(f"\nProcessing {leaf_name}...")

    # Apply flips
    leaf_dataset.flip_normals_for_timepoints(leaf_name, flip_indices)

    # Visualize to verify
    leaf_seq = leaf_dataset.get_timeseries_by_sequence_name(leaf_name)
    visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_name}_corrected")

    # Save to plant
    leaf_dataset.save_leaf_to_plant_ply(leaf_name, output_dir=output_dir)

print("\n✓ All corrections applied and saved!")

In [ ]:
# Verify final plant with all corrections
from plant_shape_analysis import PlantSequencesDataset
from plant_shape_analysis.vis.plot_functions import visualize_plant_sequence

# Reload plant dataset from corrected PLY files
plant_dataset_corrected = PlantSequencesDataset(
    dataset_path,
    use_ply=True,
    version="v2",
    alignment_method=None,
    estimate_normals=False,
)

# Visualize the corrected plant
plant_name = "maize_control_plant2"  # Change to match your plant
plant_seq = plant_dataset_corrected.get_timeseries_by_sequence_name(plant_name)
visualize_plant_sequence(
    plant_seq,
    window_name=f"{plant_name}_corrected",
    spacing=100,
    normals_scale=0.2,
    show_coordinate_frame=True,
)
